In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import csv, gzip, hashlib, io, json, os, sys, time

BASE = '/content/drive/MyDrive'
MANIFEST = f'{BASE}/model_entity_metadata_mapping.csv'
OUT = f'{BASE}/manifest_extract'
os.makedirs(OUT, exist_ok=True)
LOG = f'{OUT}/run_log.txt'
for stale in ('FAILED.txt', 'DONE.json'):
    if os.path.exists(f'{OUT}/{stale}'):
        os.remove(f'{OUT}/{stale}')

def log(msg):
    line = time.strftime('%Y-%m-%d %H:%M:%S') + '  ' + str(msg)
    print(line)
    with open(LOG, 'a') as f:
        f.write(line + '\n')

def fatal(msg):
    log('FATAL: ' + msg)
    with open(f'{OUT}/FAILED.txt', 'w') as f:
        f.write(msg + '\n')
    raise RuntimeError(msg)

log('run started')
if not os.path.exists(MANIFEST):
    fatal(f'manifest not found at {MANIFEST}')
size = os.path.getsize(MANIFEST)
log(f'manifest size {size:,} bytes')
if size < 4_000_000_000:
    fatal(f'manifest is {size:,} bytes, expected about 4.58 GB; incomplete copy?')

In [ ]:
try:
    import polars as pl
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'polars'], check=True)
    import polars as pl
log(f'python {sys.version.split()[0]}, polars {pl.__version__}')

with open(MANIFEST, 'r', newline='') as f:
    head = [next(f) for _ in range(6)]
with open(f'{OUT}/manifest_head.txt', 'w') as f:
    f.writelines(head)
cols = next(csv.reader([head[0]]))
json.dump(cols, open(f'{OUT}/manifest_columns.json', 'w'), indent=1)
log(f'{len(cols)} columns: {cols}')
for c in ('uniprotAccession', 'taxId', 'ipTM'):
    if c not in cols:
        fatal(f'required column {c!r} not in the manifest header')
for line in head[1:]:
    log('row: ' + line.strip()[:400])

In [ ]:
COPIES = '''A0A024R713 A0A024RDL8 A0A061I4L2 A0A096MRG7 A0A096N420 A0A096N9L4 A0A096NJ78 A0A096NS90 A0A096NXR7 A0A096NYC1 A0A0A0MVM9 A0A0D9QVB9 A0A0D9QX75 A0A0D9QXH2 A0A0D9QYT8 A0A0D9R0S6 A0A0D9R260 A0A0D9R909 A0A0D9R9P4 A0A0D9RBZ3 A0A0D9RD62 A0A0D9RH98 A0A0D9RIA4 A0A0D9RKX8 A0A0D9RP64 A0A0D9RUT4 A0A0D9S096 A0A0D9S7X7 A0A0D9S820 A0A0P6J1U6 A0A0P6J5R3 A0A0P6J801 A0A0P6K3Y0 A0A0P6K622 A0A0Q3LWQ1 A0A0Q3WNT4 A0A0S2Z4E1 A0A0S2Z4J6 A0A0S2Z5F5 A0A0S2Z5K9 A0A0S2Z5L0 A0A0S2Z633 A0A140TAX9 A0A140VJN9 A0A140VJT9 A0A140VJU5 A0A140VK19 A0A140VKB3 A0A151MRC4 A0A1A6HNH6 A0A1D5QQW9 A0A1D5QT80 A0A1D5RCQ6 A0A1L1RX26 A0A1L8FEA0 A0A1S2ZWZ2 A0A1S3ANB6 A0A1S3ASI5 A0A1S3F3N5 A0A1S3FIW8 A0A1S3GEA1 A0A1U7Q700 A0A1U7QEZ4 A0A1U7QFZ6 A0A1U7QPM2 A0A1U7REN2 A0A1U7SL43 A0A1U7T3H9 A0A1U7TG56 A0A1U7UBR4 A0A1U7UI06 A0A1U7USX8 A0A1U7V3L7 A0A1U8D0J7 A0A212CDY7 A0A218UC43 A0A218UGQ0 A0A226MCU9 A0A226N3W2 A0A250XUW6 A0A250XXK7 A0A250YAH9 A0A250YCC0 A0A286XDE8 A0A286XPD7 A0A286XPP9 A0A286Y0I9 A0A286ZL95 A0A287AVV3 A0A287D865 A0A2I2Y6S6 A0A2I2YBL5 A0A2I2YG18 A0A2I2YMN4 A0A2I2YRJ6 A0A2I2YS93 A0A2I2Z415 A0A2I2ZF53 A0A2I3GCB9 A0A2I3GDW6 A0A2I3HPG5 A0A2I3HVT1 A0A2I3HZ59 A0A2I3HZY2 A0A2I3LFQ8 A0A2I3LIX7 A0A2I3LNK3 A0A2I3LZP2 A0A2I3MUN4 A0A2I3MWW5 A0A2I3NAD4 A0A2I3RJX1 A0A2I3RKD8 A0A2I3RPH6 A0A2I3RVQ9 A0A2I3RX08 A0A2I3RZV5 A0A2I3S0A0 A0A2I3S6J4 A0A2I3SAK0 A0A2I3SLA8 A0A2I3T0V5 A0A2I3T1E9 A0A2I3T1Z2 A0A2I3T9Y4 A0A2I3TB90 A0A2I3TKQ4 A0A2I3TTM6 A0A2J8K3L2 A0A2J8L0A3 A0A2J8LD58 A0A2J8RGJ6 A0A2J8RXX7 A0A2J8SS55 A0A2J8U7T5 A0A2J8UBU4 A0A2J8W8W8 A0A2J8WM09 A0A2J8X957 A0A2K5BX56 A0A2K5BY65 A0A2K5C0P6 A0A2K5C946 A0A2K5CDF5 A0A2K5CH33 A0A2K5CRS7 A0A2K5D3K0 A0A2K5DPI8 A0A2K5DZH5 A0A2K5E2U4 A0A2K5EIX0 A0A2K5F5P9 A0A2K5F7E5 A0A2K5HD64 A0A2K5HED9 A0A2K5HIN9 A0A2K5HKQ9 A0A2K5HLR8 A0A2K5HTF2 A0A2K5ICA7 A0A2K5IHJ6 A0A2K5J672 A0A2K5JCR1 A0A2K5JDK3 A0A2K5JIP1 A0A2K5K1D8 A0A2K5K1H0 A0A2K5K9B6 A0A2K5KFQ7 A0A2K5KLZ3 A0A2K5LLC0 A0A2K5LTB4 A0A2K5LW89 A0A2K5M316 A0A2K5M3N9 A0A2K5MJ91 A0A2K5MKM8 A0A2K5MS99 A0A2K5MSR4 A0A2K5NEJ9 A0A2K5NFQ3 A0A2K5NG06 A0A2K5NIM4 A0A2K5NR44 A0A2K5NY11 A0A2K5P5V0 A0A2K5PJJ2 A0A2K5PJR4 A0A2K5PWS1 A0A2K5Q2T4 A0A2K5QAH6 A0A2K5QEH0 A0A2K5QGY1 A0A2K5RA84 A0A2K5RF74 A0A2K5S5H8 A0A2K5S6F6 A0A2K5S737 A0A2K5SGU1 A0A2K5TLW8 A0A2K5TNF7 A0A2K5UN07 A0A2K5UVL2 A0A2K5V7T2 A0A2K5V8I8 A0A2K5VW01 A0A2K5WQL2 A0A2K5WYH8 A0A2K5X008 A0A2K5X5D7 A0A2K5XEY3 A0A2K5XM19 A0A2K5XNV3 A0A2K5XPH4 A0A2K5YIJ9 A0A2K5YQT8 A0A2K5ZA38 A0A2K5ZGR5 A0A2K5ZN67 A0A2K5ZP40 A0A2K5ZSS3 A0A2K6A3T9 A0A2K6A5F1 A0A2K6AA78 A0A2K6AL17 A0A2K6B0D0 A0A2K6B0V7 A0A2K6B459 A0A2K6BAP7 A0A2K6BNN7 A0A2K6BSH2 A0A2K6CC68 A0A2K6CCF9 A0A2K6CKV5 A0A2K6CNN1 A0A2K6CRA3 A0A2K6DGI2 A0A2K6DKK6 A0A2K6DSR7 A0A2K6DZA0 A0A2K6EWV8 A0A2K6FNC0 A0A2K6FVV2 A0A2K6GQN6 A0A2K6GQT0 A0A2K6GRC0 A0A2K6GXE8 A0A2K6H088 A0A2K6K0S9 A0A2K6K2Y7 A0A2K6KTY1 A0A2K6L563 A0A2K6L6M9 A0A2K6L752 A0A2K6LXS1 A0A2K6M3P7 A0A2K6MBK9 A0A2K6MC50 A0A2K6MKJ7 A0A2K6MPN6 A0A2K6MZ23 A0A2K6NAB6 A0A2K6NSM4 A0A2K6P885 A0A2K6PK86 A0A2K6PLQ7 A0A2K6PXT7 A0A2K6Q1D9 A0A2K6Q8Z0 A0A2K6QDU3 A0A2K6QFX0 A0A2K6QNG6 A0A2K6QQ73 A0A2K6QRS3 A0A2K6R526 A0A2K6RBF1 A0A2K6RGE5 A0A2K6RGH6 A0A2K6RHG3 A0A2K6RY81 A0A2K6S5L8 A0A2K6SVA8 A0A2K6SX02 A0A2K6SZ84 A0A2K6T9L8 A0A2K6TJU2 A0A2K6TVJ8 A0A2K6U3Q1 A0A2R8P5X3 A0A2R8PEJ5 A0A2R8ZHD7 A0A2R8ZKI4 A0A2R8ZKT7 A0A2R8ZLB0 A0A2R8ZMT4 A0A2R8ZMV4 A0A2R8ZV50 A0A2R8ZVS4 A0A2R9A3C8 A0A2R9A862 A0A2R9AF90 A0A2R9AJK1 A0A2R9AQP0 A0A2R9B1A0 A0A2R9B5T6 A0A2R9B6C0 A0A2R9B8R1 A0A2R9B8U9 A0A2R9BBS1 A0A2R9BCG1 A0A2R9BCQ8 A0A2R9BCZ4 A0A2R9BEY4 A0A2R9BGP5 A0A2R9BH85 A0A2R9BIA4 A0A2R9BMQ7 A0A2R9BX16 A0A2R9BZF8 A0A2R9C062 A0A2R9C092 A0A2R9C1W0 A0A2R9C3Z3 A0A2R9C489 A0A2R9C564 A0A2R9C5S7 A0A2R9C630 A0A2R9C869 A0A2R9CEC8 A0A2R9CH97 A0A2R9CHQ5 A0A2R9CJ69 A0A2R9CLF8 A0A2R9CM76 A0A2U3V1X3 A0A2U3V6T7 A0A2U3VJ21 A0A2U3VM21 A0A2U3VMV4 A0A2U3VQZ7 A0A2U3VRL2 A0A2U3WTH7 A0A2U3X1J8 A0A2U3X252 A0A2U3XLH6 A0A2U3XV15 A0A2U3Z4B1 A0A2U4AAF0 A0A2U4AGG3 A0A2U4BAD4 A0A2U4CQB4 A0A2Y9DE19 A0A2Y9DU73 A0A2Y9E1I1 A0A2Y9E3F1 A0A2Y9F149 A0A2Y9GGQ9 A0A2Y9GPM4 A0A2Y9H0Q1 A0A2Y9I372 A0A2Y9I5L8 A0A2Y9I9A1 A0A2Y9IQY4 A0A2Y9IRY1 A0A2Y9ISQ7 A0A2Y9IVZ8 A0A2Y9K325 A0A2Y9KN98 A0A2Y9L2T8 A0A2Y9LDX1 A0A2Y9MAR0 A0A2Y9MUS8 A0A2Y9N3P9 A0A2Y9N4Y0 A0A2Y9NGT6 A0A2Y9NZH3 A0A2Y9PFG4 A0A2Y9PG95 A0A2Y9QJJ8 A0A2Y9SJ83 A0A2Y9TEP8 A0A2Y9TK02 A0A340WCB5 A0A340X6J6 A0A340XJ72 A0A340Y8M2 A0A340YFH9 A0A341AGW4 A0A341BZP7 A0A341C338 A0A383YRB4 A0A383ZAV4 A0A384A1P3 A0A384AAN6 A0A384BA19 A0A384C5Q3 A0A384CE78 A0A384CIN9 A0A384CLG6 A0A384MDR8 A0A384MR07 A0A384MTQ8 A0A384N5Z8 A0A384NL29 A0A384NY87 A0A3L8RX38 A0A3M0IX44 A0A3M0JPL7 A0A3Q2L5Q4 A0A3Q7MM67 A0A3Q7MY35 A0A3Q7N8I2 A0A3Q7NWM5 A0A3Q7P3K4 A0A3Q7PB80 A0A3Q7PJ45 A0A3Q7PZ40 A0A3Q7RZH4 A0A3Q7SKY2 A0A3Q7T8E2 A0A3Q7T8M4 A0A3Q7TSE2 A0A3Q7TY84 A0A452DQG7 A0A452E0V0 A0A452F0Y5 A0A452FNB9 A0A452FX55 A0A452G120 A0A452G164 A0A452HSW5 A0A452I2W5 A0A452ISN9 A0A452R7G6 A0A452RUK6 A0A452RVP0 A0A455BS64 A0A480TKR3 A0A484GL78 A0A485NPC4 A0A485NRQ2 A0A485NUH3 A0A485P3Z7 A0A485P8B8 A0A485PSH6 A0A4D9DPZ8 A0A4D9ETQ5 A0A4U1EQW4 A0A4W2CJ58 A0A4W2DCG2 A0A4W2EGD4 A0A4W2ELK4 A0A4W2ELV6 A0A4W2FMN1 A0A4W2G4L3 A0A4W2HTY6 A0A4X2KPV0 A0A5E4AG38 A0A5E4ARE7 A0A5E4CUI0 A0A5E4CVV7 A0A5F7ZE66 A0A5F7ZR94 A0A5F7ZV58 A0A5F8A4H3 A0A5F8GNU3 A0A5F9D4Q9 A0A5F9D8G4 A0A5J5MQ27 A0A5J5N5S4 A0A5N3VKY7 A0A5N3VR14 A0A5N4DST4 A0A643C0Y2 A0A663E050 A0A663F498 A0A663LXS5 A0A667GDG9 A0A667GF51 A0A667GM19 A0A667GRK8 A0A667HMC6 A0A667IAT8 A0A667IB03 A0A667IDC6 A0A669PS50 A0A669QRI6 A0A670HLV0 A0A670JEY6 A0A670YJD1 A0A670ZFT6 A0A671DI96 A0A671E925 A0A671ESP1 A0A671F079 A0A671FUJ7 A0A671G2H2 A0A672U6Z4 A0A672UAJ9 A0A672UND9 A0A673T7L2 A0A673T7S6 A0A673UGA3 A0A673UJ70 A0A673UMU0 A0A673UQY0 A0A673VS39 A0A674H8Z6 A0A674IM16 A0A674JXK7 A0A674K7L0 A0A6B0RN98 A0A6B0SB27 A0A6D2WLT4 A0A6D2WU23 A0A6D2XXA8 A0A6D2Y0E5 A0A6G1ASV2 A0A6I9HX03 A0A6I9HXJ1 A0A6I9INX0 A0A6I9L7K4 A0A6I9MCL6 A0A6I9MGY2 A0A6I9MJW2 A0A6I9Y6D6 A0A6I9YCV5 A0A6I9ZFK9 A0A6I9ZGE2 A0A6I9ZZX1 A0A6J0H9I6 A0A6J0I5X3 A0A6J0TM64 A0A6J0V5F8 A0A6J0VFQ5 A0A6J0XZX1 A0A6J0Y9B7 A0A6J0YHM1 A0A6J0YQ89 A0A6J1ULT1 A0A6J1VR58 A0A6J1XGZ7 A0A6J1Y7N9 A0A6J1YLD2 A0A6J1YMB1 A0A6J1ZEH0 A0A6J1ZW24 A0A6J2AVR7 A0A6J2B1X6 A0A6J2B7P7 A0A6J2BJ23 A0A6J2D641 A0A6J2E1H0 A0A6J2EN73 A0A6J2F9Q9 A0A6J2FVZ2 A0A6J2FWN6 A0A6J2HLT9 A0A6J2MZD8 A0A6J2N084 A0A6J2N3A4 A0A6J2NE46 A0A6J3DSD9 A0A6J3DYH9 A0A6J3EWI9 A0A6J3EY93 A0A6J3F928 A0A6J3FFW6 A0A6J3FFZ9 A0A6J3FKU5 A0A6J3FYE5 A0A6J3GC21 A0A6J3GDQ9 A0A6J3GHM9 A0A6J3GVA0 A0A6J3HP55 A0A6J3IBY7 A0A6J3IR04 A0A6J3IXV9 A0A6J3JRF1 A0A6J3PTY8 A0A6J3R623 A0A6P3EUR0 A0A6P3FHZ7 A0A6P3FN68 A0A6P3FQV7 A0A6P3GKU2 A0A6P3GNP9 A0A6P3HW21 A0A6P3HYI0 A0A6P3I0X6 A0A6P3IXC9 A0A6P3PXJ2 A0A6P3PY75 A0A6P3QH67 A0A6P3QI90 A0A6P3QX32 A0A6P3RDY9 A0A6P3RJQ5 A0A6P5C4Q1 A0A6P5D1U3 A0A6P5DBW3 A0A6P5L8Z9 A0A6P5LX96 A0A6P5NVP0 A0A6P5PVK8 A0A6P5Q1R2 A0A6P5R6B7 A0A6P6H2V6 A0A6P6HJJ3 A0A6P6ICY6 A0A6P6INM8 A0A6P6ISH9 A0A6P7QXH4 A0A6P9CVZ2 A0A6P9DHA3 A0A7J7F7Z4 A0A7J7S3I3 A0A7J7UHH0 A0A7J7UTC3 A0A7J7XAQ1 A0A7J7XFB2 A0A7J7Z325 A0A7J7ZIE4 A0A7J8AJX4 A0A7J8AZU7 A0A7J8BUC7 A0A7J8C0G1 A0A7J8CKV9 A0A7J8FGC6 A0A7J8FX62 A0A7J8GXX7 A0A7J8H9F0 A0A7J8I5J5 A0A7J8IKT5 A0A7J8JF27 A0A7J8JP39 A0A7J8JVA2 A0A7J8KER1 A0A7K4Q0A6 A0A7K5F605 A0A7K6B409 A0A7K6TF43 A0A7K7NF55 A0A7K9HR73 A0A7K9SED6 A0A7L0F6F9 A0A7L1P4Z0 A0A7L3R950 A0A7L3UGB0 A0A7L3XHM0 A0A7L4C7I2 A0A7L4FFQ9 A0A7L4GIE7 A0A7M4DWP6 A0A7M4DXV1 A0A7N4NPW3 A0A7N4PL13 A0A7N5JB11 A0A7N5JD27 A0A7N5JJ02 A0A7N5JN20 A0A7N5K2N8 A0A7N5KEP5 A0A7N9C7T0 A0A811YII7 A0A811YP31 A0A811ZGK4 A0A850XES7 A0A851KGB5 A0A851RGC0 A0A851V107 A0A851VUS1 A0A851XE64 A0A851Y3I7 A0A852BSZ6 A0A852GM07 A0A852IAP9 A0A852KN01 A0A8B6Y5H7 A0A8B6Y5J6 A0A8B6YLC6 A0A8B6YSW5 A0A8B6ZJT2 A0A8B6ZT90 A0A8B7A9E7 A0A8B7F1Y9 A0A8B7FED9 A0A8B7FIY9 A0A8B7FQ00 A0A8B7G7A2 A0A8B7HBQ8 A0A8B7JSA8 A0A8B7QCA4 A0A8B7QCC5 A0A8B7RPW1 A0A8B7RXU5 A0A8B7SUY6 A0A8B7SYY7 A0A8B7T8N9 A0A8B7TVU8 A0A8B7TWU5 A0A8B7WMB8 A0A8B8RVG5 A0A8B8RWV3 A0A8B8SEY4 A0A8B8V216 A0A8B8V396 A0A8B8WTI3 A0A8B8WXN0 A0A8B8YFM0 A0A8B8YT79 A0A8B9EUI9 A0A8B9G066 A0A8B9G1D7 A0A8B9NG62 A0A8B9NLT4 A0A8B9RP69 A0A8B9S4U0 A0A8B9WGN4 A0A8B9X382 A0A8B9XNN9 A0A8B9Y3M3 A0A8B9YM05 A0A8B9YSF6 A0A8B9YUX1 A0A8C0AAM1 A0A8C0CHT2 A0A8C0GG69 A0A8C0I0M9 A0A8C0IKJ7 A0A8C0JXF0 A0A8C0JZ25 A0A8C0KDH9 A0A8C0LHD3 A0A8C2M688 A0A8C2MYC2 A0A8C2U607 A0A8C2VFM2 A0A8C2VR70 A0A8C2W2P4 A0A8C2W5R4 A0A8C2W7G0 A0A8C2YB66 A0A8C2YMW1 A0A8C3BQ08 A0A8C3FUJ9 A0A8C3HKW4 A0A8C3I9F9 A0A8C3K937 A0A8C3L6W2 A0A8C3MT53 A0A8C3NBY5 A0A8C3NWH4 A0A8C3RAV4 A0A8C3SB52 A0A8C3SCZ2 A0A8C3ST93 A0A8C3TVQ4 A0A8C3TXF3 A0A8C3VQ87 A0A8C3VQP0 A0A8C3YE26 A0A8C4FHV3 A0A8C4JV66 A0A8C4KSE0 A0A8C4L5Q1 A0A8C4MRF8 A0A8C4PGW8 A0A8C4U862 A0A8C4VQN9 A0A8C4WF31 A0A8C4YBF3 A0A8C5IG50 A0A8C5J6C7 A0A8C5JGV9 A0A8C5K3L4 A0A8C5K7U1 A0A8C5SXC9 A0A8C5TBQ3 A0A8C5TDL8 A0A8C5W2F9 A0A8C5XNN5 A0A8C5YNL3 A0A8C5Z3P1 A0A8C5ZC11 A0A8C6A1K7 A0A8C6A7B8 A0A8C6ABZ2 A0A8C6AEK2 A0A8C6AT23 A0A8C6AVP6 A0A8C6AWI6 A0A8C6CDL0 A0A8C6CK68 A0A8C6CU89 A0A8C6D571 A0A8C6DCN0 A0A8C6DQR0 A0A8C6ERQ0 A0A8C6FC07 A0A8C6FVV5 A0A8C6GVU6 A0A8C6HIG5 A0A8C6MYE8 A0A8C6N4U1 A0A8C6NF85 A0A8C6QMK1 A0A8C6QZP2 A0A8C6R5V6 A0A8C6W142 A0A8C6XMC7 A0A8C6YHA4 A0A8C6ZKR6 A0A8C7AML1 A0A8C7ASI3 A0A8C7B4W4 A0A8C7BRB4 A0A8C7BXR6 A0A8C8SQW3 A0A8C8SXH9 A0A8C8TYT7 A0A8C8VKJ2 A0A8C8WK10 A0A8C8WTG7 A0A8C8X058 A0A8C8XJN6 A0A8C8Y5X7 A0A8C8YNW7 A0A8C8YYE6 A0A8C8ZKS2 A0A8C8ZNQ1 A0A8C9A241 A0A8C9C087 A0A8C9C1E0 A0A8C9C6B2 A0A8C9CHJ9 A0A8C9CKG2 A0A8C9CTA3 A0A8C9D788 A0A8C9DB96 A0A8C9E0R1 A0A8C9GG47 A0A8C9GGG5 A0A8C9GJK9 A0A8C9GJN5 A0A8C9GNH7 A0A8C9GZL6 A0A8C9H5B7 A0A8C9I2L1 A0A8C9I8D6 A0A8C9IEA0 A0A8C9IFD9 A0A8C9ISD6 A0A8C9IUM5 A0A8C9KGF8 A0A8C9KI96 A0A8C9KMD9 A0A8C9KQ21 A0A8C9KR80 A0A8C9LBP8 A0A8C9LS84 A0A8C9LSD2 A0A8C9LU09 A0A8C9P5Y6 A0A8C9QNE0 A0A8C9QS45 A0A8D0BHR0 A0A8D0C085 A0A8D0HJ18 A0A8D0KNU2 A0A8D2AN86 A0A8D2B0G5 A0A8D2CP13 A0A8D2CTG9 A0A8D2CX98 A0A8D2DDC8 A0A8D2DSD9 A0A8D2DSM3 A0A8D2E4G5 A0A8D2E6E2 A0A8D2E9C0 A0A8D2EBK4 A0A8D2EFL6 A0A8D2EG38 A0A8D2EK92 A0A8D2ESM2 A0A8D2ETS4 A0A8D2EVD7 A0A8D2FCB4 A0A8D2FJE8 A0A8D2FQZ6 A0A8D2FRD1 A0A8D2FTK1 A0A8D2G1T6 A0A8D2G2D3 A0A8D2G6N5 A0A8D2GKM5 A0A8D2GN31 A0A8D2GRT8 A0A8D2H0S3 A0A8D2HSI7 A0A8D2I767 A0A8D2ICG3 A0A8D2IHH6 A0A8D2IIR4 A0A8D2ISV9 A0A8D2IV63 A0A8D2JKV1 A0A8D2JRU9 A0A8D2JVR6 A0A8D2K493 A0A8D2N2F9 A0A8I3B288 A0A8I3N9W4 A0A8I3PCV3 A0A8I3PYF3 A0A8I3QBB4 A0A8I3VY59 A0A8I3W3H6 A0A8I3X7L1 A0A8I4A6E1 A0A8I5NGI6 A0A8I5NNS2 A0A8I5NX06 A0A8J5ZKY8 A0A8J6GAA2 A0A8K1GAD1 A0A8K1LEC4 A0A8M1F0D3 A0A8M1FRX7 A0A8M1G2T4 A0A8M1GPR3 A0A8M1H3B8 A0A8M1MJP6 A0A8T1S145 A0A8U0NFL4 A0A8U0RJN9 A0A8U0RPS8 A0A8U0SP93 A0A8U0T7E9 A0A8U0V5A8 A0A8V5GKW4 A0A9B0TNH4 A0A9B0TSJ6 A0A9B0U7P2 A0A9B0U8P1 A0A9D3XIA4 A0A9D4ARU4 A0A9F2WHM8 A0A9J7FC62 A0A9J7G105 A0A9L0JVM3 A0A9L0KC47 A0A9L0RH86 A0A9L0RQW6 A0A9Q0XFH5 A0A9Q0XK12 A0A9Q0Y3Z1 A0A9V1EEH5 A0A9V1EFW8 A0A9V1ERY2 A0A9V1ES84 A0A9V1ESW6 A0A9V1EZ07 A0A9V1F0Q0 A0A9V1FQ84 A0A9V1FS23 A0A9W2V4R6 A0A9W3EFM8 A0A9W3F6X9 A0A9W3GZ51 A0A9X9LJ29 A0A9X9PY33 A0AA35K0G7 A0AA35KW98 A0AA35LHF2 A0AA35LJC4 A0AA41MTF2 A0AA41N6I8 A0AA41NAK7 A0AA97K6H3 A0AAA9RVP0 A0AAA9T1A1 A0AAA9TPN5 A0AAD4TY75 A0AAD4U2Y9 A0AAD4UIZ4 A0AAD4UKP5 A0AAD4Y7M3 A0AAD4YBF8 A0AAD4YFF5 A0AAN7NST4 A0AAV0ACX6 A0AAW0H3E3 A0AAW0HBR1 A0AAX6NTI0 A0AAX6NV18 A0AB34HDN1 A0ABC9VXC1 A0ABC9XI35 A0ABD2D8H5 A0ABD2DNS8 A0ABD2ED84 A0ABD2EKV9 A0ABI7WB83 A0ABI7YSP6 A0ABI7Z092 A0ABI7Z0R9 A0ABI8A251 A0ABI8AEI5 A0ABM0H5T2 A0ABM0HE05 A0ABM0HG19 A0ABM0HK27 A0ABM0HKJ4 A0ABM0I253 A0ABM0I3Q1 A0ABM0I878 A0ABM0J986 A0ABM0KBT8 A0ABM0LMT2 A0ABM0LP67 A0ABM0Q7P4 A0ABM0QRL7 A0ABM0RMN5 A0ABM0S0C6 A0ABM0SJU9 A0ABM1AYZ6 A0ABM1JLS8 A0ABM1U0X4 A0ABM2EHZ1 A0ABM2FEM0 A0ABM2X7H5 A0ABM3RZC5 A0ABM3TVJ4 A0ABM3WDH1 A0ABM3XNV5 A0ABM4F7E0 A0ABM4FEL9 A0ABM4GUE5 A0ABM4HX39 A0ABM4ICT0 A0ABM4IVH3 A0ABM4JDB7 A0ABM4KE66 A0ABM4KFB6 A0ABM4MBT7 A0ABM4N002 A0ABM4QNT8 A0ABM4QS23 A0ABM4QV48 A0ABM4SC29 A0ABM4SEK8 A0ABM4SYG5 A0ABM4TJ88 A0ABM4XHP1 A0ABM4XNA7 A0ABM5B3P8 A0ABM5C4U2 A0ABM5C7R4 A0ABM5CLV8 A0ABM5CNQ6 A0ABN8XVK4 A0ABN8YCC7 A0ABN8YJ72 A0ABN8YUY3 A0ABN8YXI5 A0ABN8Z9F8 A0ABN9ZRK8 A0ABN9ZS62 A0ABN9ZZL6 A0ABQ0EX43 A0ABQ0FLK5 A0ABQ0FQN9 A0ABQ9DNR5 A0ABQ9UA75 A0ABQ9UWH8 A0ABQ9W4P2 A0AC58KF50 A0AC58LKX5 A0AC58P799 A0AC58P9P5 A0AC58PCX7 A0AC58PRR2 A0AC58QTW6 A0ACB9UJ94 A0ACB9UU99 A0ACB9V329 A0ACB9V926 A0ACB9VB38 A0ACB9VF19 A0ACE2UWE7 A0ACE2V230 A0ACE2V5Y6 A0ACE2WUG8 A0ACE2ZMV7 A0ACE3AQ82 A0ACF6FS81 A0ACF6FWA8 A0ACF6H4V2 A0ACF6H7H0 A0ACF8HCP5 A0ACF8HF62 A0ACF8HGR0 A0ACF8HKP5 A0ACF8I556 A0ACF8IAJ3 A0ACF8IC16 A0ACF8IIP0 A0ACF8J3G7 A0ACF8J621 A0ACF8J6F3 A0ACF8JA26 A0ACF8JCJ0 A0ACF8JI58 A0ACF8JTU8 A0ACF8JVD8 A0ACF8K9D6 A0ACF8KMA3 A0ACF8KMY4 A0ACF8N8R5 A0ACF8NBL9 A0ACF8NMX7 A0ACF8NS90 A0ACF8NZR7 A0ACF8P9T3 A0ACF8P9U4 A0ACF8PD24 A0ACF8PJ67 A0ACF8PPK2 A0ACF8PWR8 A0ACF8Q0L7 A0ACF8Q227 A0ACI6NJL3 A0ACI6NVB1 A0ACI6P2C6 A0ACI6QBQ7 A0ACI6QLU8 A0ACI6RQC1 A0ACI6RTL7 A0ACM8QFX4 A0ACM8QR60 A0ACM8QS90 A0ACM8QTE1 A0ACM8QV09 A0ACM8QWC5 A0ACM8QWP7 A0ACM8QY47 A0ACM8R321 A0ACM8R425 A0ACM8R4N3 A0ACM8R4U9 A0ACM8R552 A0ACM8R5Q9 A0ACM8R6I6 A0ACM8R6M0 A0ACM8R7I4 A0ACM8R7Y1 A0ACM8R9H0 A0ACM8RDL5 A4FU51 A4FUE1 A5PJR5 A5PKX1 A8K3L7 A8K9L3 A8KAF9 B0YJ56 B8Y648 C4B4C6 D2IYK3 D3Z8N4 D3Z8W5 E1C2J6 F1SDA0 F2Z505 F6PFV0 F6PU05 F6RJT4 F6RMP8 F6WAT1 F6WFV1 F7A3C2 F7AAN6 F7BUE9 F7CTK0 F7FI72 F7FJH3 F7FU88 F7G8M9 F7GMG2 F7GQY1 F7HVH6 F7I1T5 F7I7Z9 F7IF11 G1M0L3 G1MAT4 G1MDR1 G1NN47 G1QH46 G1QJH1 G1QSD2 G1QUX6 G1QXE0 G1REZ0 G1RTB6 G1S0E7 G1S103 G1S7V0 G1S8G2 G1SJW7 G2HF91 G2HH14 G2HHE6 G3I1V1 G3QCU8 G3QFL8 G3QW07 G3QWG9 G3QXL5 G3QYE2 G3QZW6 G3R4K1 G3R6I5 G3R8Q2 G3RBH2 G3RG93 G3RHI3 G3RHS5 G3RJK4 G3RK50 G3RMN7 G3RPG2 G3RTU9 G3RVY8 G3RXB8 G3S2Z8 G3S5I4 G3S9M3 G3SAQ5 G3SGL3 G3SHP0 G3SJP8 G3TH63 G7MFD2 G7NHF8 G7NQC4 G7NT75 G7NVI1 G7NWV0 G7PBX8 G7Q3I8 G9BWQ3 H0UYD7 H0WHC6 H0WW71 H0XCI0 H0XYI0 H2N816 H2N859 H2NDR7 H2NHB0 H2NJL0 H2NPJ0 H2NQT3 H2NR44 H2NW52 H2NYW5 H2P4P6 H2PGP3 H2PQF5 H2PY21 H2PYG2 H2PZF1 H2PZI0 H2Q0M8 H2Q3D6 H2Q484 H2Q793 H2Q917 H2QAV7 H2QBZ4 H2QEF0 H2QFU2 H2QHM3 H2QJR9 H2QJU8 H2QMV0 H2QNK4 H2QP96 H2QVD0 H2QVV4 H2QW84 H2QWI9 H2QX44 H2QZX4 H2R176 H2RAA1 H2RBR9 H9EM41 H9EQM3 H9FNA9 H9FNK4 H9FUZ4 H9FZA6 H9YWA6 H9Z160 H9ZB28 I0FIL8 I0FLA6 I0FSI6 I3LYE1 I3M011 I3M3J2 I3MFT2 I3MKS4 I3MPF4 I3MRA1 I3ND19 I3NEJ2 I7GH11 J3S1H7 J3S4U3 K4PZ75 K7A1E7 K7A1J8 K7A3W1 K7AJC4 K7ATS5 K7ATW1 K7AUU6 K7AX09 K7AY87 K7AZV1 K7B607 K7BAP8 K7BB92 K7BH92 K7BNE8 K7BPE0 K7BRS8 K7BVH2 K7BX03 K7BXE1 K7C140 K7C180 K7C6M9 K7C8W3 K7CAE8 K7CBA5 K7CDV2 K7CH04 K7CJE6 K7CMB7 K7CQ53 K7CU20 K7CZP2 K7D0Y1 K7D343 K7D364 K7D4Z1 K7DAK7 K7DCK5 K7DNB3 K7DNU9 K7DSD6 K7FLQ7 K7PPU3 K7WQT1 K9IHD4 K9IJW5 L5JUS2 L5KAT7 L5KYE3 L5L209 L8IMF5 M1EN77 M3Y007 M3YD56 M3YVV7 M3Z1V3 M3ZB09 M7AI21 P0DMN0 P50224-1 P60175 P60202 P60202-1 P60203 P60518 P61017 P62496 P62497 P63163 P63164 P68037 Q08DK3 Q0II28 Q0VC56 Q0VCX5 Q0VGK0 Q1ET61 Q25QJ5 Q3MHP1 Q3UN87 Q3UYM8 Q4AEH2 Q4AEH2-1 Q4R567 Q53XC6 Q561N4 Q5BIZ2 Q5R6E6 Q5R6E6-1 Q5R6I0 Q5R8P5 Q5RAR0 Q5RCV8 Q5U0Q1 Q5U2Q7 Q5ZKN7 Q60HD3 Q6FGU2 Q6GQ27 Q6IB04 Q712P7 Q8BWY3 Q8HYB6 Q8R3R8 Q8WV14-3 R7VNC7 S7P9S9 T1D7D6 T1E4T3 U3BPG0 U3CAC4 U3CV73 U3D9Y3 U3DBU1 U3E5W5 U3E5Z3 U3E808 U3EZM5 U3F5R9 U3F8Z2 U3FBV5 U3FKH4 U3FM72 U6CSS2 U6CUW9 V9HW50 V9HWC3 V9HWF3 V9HWG2 V9HWK1 V9HWN8 X5D8Y5 X5DP00 X5DQN2'''.split()
if len(COPIES) != 1461 or len(set(COPIES)) != 1461:
    fatal(f'copy list has {len(COPIES)} entries, expected 1,461 unique accessions')
log(f'{len(COPIES)} copy accessions loaded')

In [ ]:
HUMAN = ['9606', '9606.0']
t0 = time.time()
lf = pl.scan_csv(MANIFEST, infer_schema_length=0, ignore_errors=True)
q = lf.filter(pl.col('taxId').str.strip_chars().is_in(HUMAN) | pl.col('uniprotAccession').str.strip_chars().is_in(COPIES))
try:
    sub = q.collect(engine='streaming')
except TypeError:
    sub = q.collect(streaming=True)
log(f'filtered rows: {sub.height:,} in {time.time() - t0:.0f} s')
if sub.height < 50_000:
    fatal(f'only {sub.height:,} rows kept; expected about 91,641 human rows with ipTM')

human = sub.filter(pl.col('taxId').str.strip_chars().is_in(HUMAN))
cop = sub.filter(pl.col('uniprotAccession').str.strip_chars().is_in(COPIES))
log(f'human rows {human.height:,}; copy rows {cop.height:,}')
n_iptm = human.filter(pl.col('ipTM').is_not_null() & (pl.col('ipTM').str.strip_chars() != '')).height
log(f'human rows with ipTM: {n_iptm:,} (expected 91,641)')

def write_tsv_gz(df, path):
    buf = io.BytesIO()
    df.write_csv(buf, separator='\t')
    raw = buf.getvalue()
    with gzip.open(path, 'wb', compresslevel=6) as g:
        g.write(raw)
    return hashlib.md5(raw).hexdigest()

md5_h = write_tsv_gz(human, f'{OUT}/manifest_human.tsv.gz')
md5_c = write_tsv_gz(cop, f'{OUT}/manifest_copies.tsv.gz')
log('wrote manifest_human.tsv.gz and manifest_copies.tsv.gz')

In [ ]:
expected = {'O95671': 0.60, 'O95671-3': 0.65, 'Q8NDH6': 0.54, 'Q8NDH6-2': 0.10}
for acc, v in expected.items():
    vals = [float(x) for x in human.filter(pl.col('uniprotAccession').str.strip_chars() == acc)['ipTM'].to_list() if x not in (None, '')]
    log(f'check {acc}: manifest ipTM values {vals}, expected {v}')
    if not any(abs(x - v) < 0.0051 for x in vals):
        fatal(f'{acc}: no manifest row has ipTM {v}; columns may be misread')

summary = {'manifest_bytes': size, 'columns': cols, 'human_rows': human.height, 'human_rows_with_ipTM': n_iptm,
           'copy_rows': cop.height, 'copy_accessions_found': cop['uniprotAccession'].n_unique(),
           'human_unique_accessions': human['uniprotAccession'].n_unique(),
           'md5_uncompressed': {'manifest_human.tsv': md5_h, 'manifest_copies.tsv': md5_c}}
if 'modelEntityId' in cols:
    per_model = human.group_by('modelEntityId').agg(pl.col('uniprotAccession').n_unique().alias('n_acc'), pl.len().alias('n_rows'))
    summary['human_models'] = per_model.height
    summary['human_models_one_accession'] = per_model.filter(pl.col('n_acc') == 1).height
    summary['human_models_two_or_more_accessions'] = per_model.filter(pl.col('n_acc') >= 2).height
    summary['rows_per_model_counts'] = {str(k): int(v) for k, v in per_model['n_rows'].value_counts().iter_rows()}
for c in cols:
    if human[c].n_unique() <= 12:
        summary[f'values_{c}'] = {str(k): int(v) for k, v in human[c].value_counts().iter_rows()}
for c in cols:
    try:
        x = human[c].cast(pl.Float64, strict=False).drop_nulls()
    except Exception:
        continue
    if x.len() > 1000 and c not in ('taxId',):
        summary[f'quantiles_{c}'] = {q: round(float(x.quantile(q)), 4) for q in (0.0, 0.05, 0.25, 0.5, 0.75, 0.95, 1.0)} | {'n': x.len()}
json.dump(summary, open(f'{OUT}/manifest_summary.json', 'w'), indent=1)
log('wrote manifest_summary.json')

In [ ]:
t0 = time.time()
try:
    n_all = lf.select(pl.len()).collect(engine='streaming').item()
except TypeError:
    n_all = lf.select(pl.len()).collect(streaming=True).item()
log(f'manifest rows: {n_all:,} ({time.time() - t0:.0f} s)')
summary['manifest_rows'] = int(n_all)
json.dump(summary, open(f'{OUT}/manifest_summary.json', 'w'), indent=1)

files = {fn: os.path.getsize(f'{OUT}/{fn}') for fn in sorted(os.listdir(OUT))}
json.dump({'finished': time.strftime('%Y-%m-%d %H:%M:%S'), 'files': files}, open(f'{OUT}/DONE.json', 'w'), indent=1)
log('DONE. Copy the manifest_extract folder into data/manifest_extract/')
for fn, b in files.items():
    print(f'  {fn:28s} {b:>12,d} bytes')

In [ ]:
import shutil
from google.colab import files
if not os.path.exists(f'{OUT}/DONE.json'):
    fatal('DONE.json missing; the extract did not finish')
zp = shutil.make_archive('/content/manifest_extract', 'zip', OUT)
files.download(zp)